# Chronos -- serve ECNet from Kaggle over a STATIC ngrok domain

Turns this session into the public inference backend for the deployed frontend.

**Why a reserved domain:** Kaggle sessions die (12 h cap, idle timeout, weekly
GPU quota). A random ngrok URL would mean rebuilding and redeploying the
frontend every single time. A *reserved* domain is permanently yours, so the
URL is identical across restarts -- the frontend is built once and never
touched again. Session dies -> rerun this notebook -> same URL, back online.

## Before the first run

1. **Session settings** (right panel): Accelerator = **GPU T4 x2**, Internet = **On**.
2. **Find your dev domain** -- [dashboard.ngrok.com](https://dashboard.ngrok.com) ->
   *Gateway -> Domains*. Every account is auto-assigned one, free and permanent,
   shaped like `cheerful-mantis-42.ngrok-free.dev`. You can't rename it on the free plan, and
   you don't need to -- it never changes, which is all this setup requires.
3. **Add Secrets** (*Add-ons -> Secrets*) so no token is ever pasted in a cell:
   - `NGROK_AUTHTOKEN` -- from the ngrok dashboard
   - `NGROK_DOMAIN` -- the domain you just reserved
4. **Attach the weights** (*Add Input -> Datasets*): the dataset holding
   `ECNet-7.pt`. Upload it once as a private Kaggle Dataset if you haven't.

Every later run is just: open notebook -> Run All -> leave the last cell running.

In [ ]:
# ========================= PREFLIGHT =========================================
# Fail here, loudly, rather than halfway through a model load.
import subprocess, sys
from pathlib import Path

gpu = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total",
                      "--format=csv,noheader"], capture_output=True, text=True)
print("GPU:", gpu.stdout.strip() or "NONE -- set Accelerator to GPU in the right panel")

ckpts = sorted(Path("/kaggle/input").rglob("*.pt")) if Path("/kaggle/input").exists() else []
print("Checkpoints found:", [str(p) for p in ckpts] or "NONE -- Add Input -> your weights dataset")

try:
    from kaggle_secrets import UserSecretsClient
    s = UserSecretsClient()
    for k in ("NGROK_AUTHTOKEN", "NGROK_DOMAIN"):
        print(f"Secret {k}:", "set" if s.get_secret(k) else "MISSING")
except Exception as e:
    print("Secrets unavailable:", e)

In [ ]:
# ========================= SOURCE + DEPS =====================================
# Clone the repo so the server code always matches main -- no copy-paste drift.
REPO = "https://github.com/trcy7/AiVideoDetection.git"
DEST = "/kaggle/working/ecnet"

import os, subprocess
if os.path.isdir(f"{DEST}/.git"):
    subprocess.run(["git", "-C", DEST, "pull", "--ff-only"], check=True)
else:
    subprocess.run(["git", "clone", "--depth", "1", REPO, DEST], check=True)

# Kaggle already ships torch/opencv/numpy/pyyaml; add only what serving needs.
!pip -q install fastapi "uvicorn[standard]" python-multipart pyngrok timm==1.0.11

%cd {DEST}/training
print("ready")

In [ ]:
# ========================= CONFIG ============================================
# <<< EDIT ME: the EXACT origin of your deployed frontend (no trailing slash).
# A mismatch here is the #1 cause of "backend is up but uploads fail" -- the
# browser blocks the response and the UI only sees a network error.
ALLOWED_ORIGINS = "https://ecnet.example.com"

# Coverage knobs. On Kaggle's T4 the defaults (48 scoring windows, GradCAM on 8)
# are comfortable -- these exist for CPU hosts. Lower them only if analyses of
# long clips start brushing the frontend's patience.
MAX_SCORE_WINDOWS = None   # None = checkpoint default (48)
MAX_CAM_WINDOWS   = None   # None = checkpoint default (8)

In [ ]:
# ========================= SERVE (blocking) ==================================
# Loads the model, opens the reserved domain, verifies the PUBLIC url returns
# JSON, then blocks. The running cell is what keeps the session from idling out.
# Interrupt it to shut down.
import kaggle_serve

kaggle_serve.serve(
    allowed_origins=ALLOWED_ORIGINS,
    max_score_windows=MAX_SCORE_WINDOWS,
    max_cam_windows=MAX_CAM_WINDOWS,
)

## When the session ends

Nothing to redeploy. Open this notebook, **Run All**, leave the last cell
running -- the URL is unchanged, so the live frontend reconnects on its own.

**What does NOT survive:** the SQLite audit trail at `/kaggle/working/ecnet.db`
is wiped with the session. Verdict history and user feedback are lost unless you
save the notebook output first (*Save Version*), which snapshots `/kaggle/working`.

**Quota:** GPU is ~30 h/week. A session caps at 12 h. Watch the usage meter in
the right panel if the demo needs to stay up.

**Bandwidth:** ngrok's free tier meters transfer. Uploaded clips dominate it, so
the practical ceiling is a demo audience, not a launch.